In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

current = Path.cwd().resolve()

ROOT = None
for folder in [current] + list(current.parents):
    if (folder / "data").exists() and (folder / "results").exists():
        ROOT = folder
        break

if ROOT is None:
    raise FileNotFoundError("Could not locate repository root.")

EWMA_FILE = ROOT / "results" / "ewma_var_output.csv"
COMBINED_FILE = ROOT / "results" / "combined_var_results.csv"
VALIDATION_DIR = ROOT / "results" / "validation" / "ewma"
FIGURE_DIR = VALIDATION_DIR / "figures"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ewma = pd.read_csv(EWMA_FILE)
ewma["Date"] = pd.to_datetime(ewma["Date"])
ewma = ewma.sort_values("Date").reset_index(drop=True)

ewma["Validation_Exception"] = ewma["Portfolio_PnL"] < -ewma["VaR_99"]

n_obs = len(ewma)
expected_rate = 0.01
expected_exceptions = n_obs * expected_rate
actual_exceptions = int(ewma["Validation_Exception"].sum())
observed_rate = actual_exceptions / n_obs

exception_summary = pd.DataFrame({
    "Metric": [
        "Backtesting Observations",
        "Expected Exceptions",
        "Actual Exceptions",
        "Expected Exception Rate",
        "Observed Exception Rate"
    ],
    "Value": [
        n_obs,
        expected_exceptions,
        actual_exceptions,
        expected_rate,
        observed_rate
    ]
})

ewma["Year"] = ewma["Date"].dt.year

annual = ewma.groupby("Year").agg(
    Observations=("Validation_Exception", "size"),
    Actual_Exceptions=("Validation_Exception", "sum")
)

annual["Expected_Exceptions"] = annual["Observations"] * expected_rate
annual["Exception_Rate"] = annual["Actual_Exceptions"] / annual["Observations"]

annual = annual[
    [
        "Observations",
        "Expected_Exceptions",
        "Actual_Exceptions",
        "Exception_Rate"
    ]
]

exceptions = ewma[ewma["Validation_Exception"]]

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    ewma["Date"],
    ewma["Portfolio_PnL"],
    color="0.40",
    linewidth=0.7,
    label="Daily Portfolio P&L"
)

ax.plot(
    ewma["Date"],
    -ewma["VaR_99"],
    linewidth=1.3,
    label="99% EWMA VaR Loss Threshold"
)

ax.scatter(
    exceptions["Date"],
    exceptions["Portfolio_PnL"],
    color="red",
    s=24,
    label=f"Exceptions ({len(exceptions)})",
    zorder=3
)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("EWMA 1-Day 99% VaR Backtesting for $1M SPY Portfolio")
ax.set_xlabel("Date")
ax.set_ylabel("Dollar P&L / VaR Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "ewma_backtesting_exceptions.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

def kupiec_test(exception_series, expected_probability=0.01):
    e = np.asarray(exception_series, dtype=int)

    T = len(e)
    x = int(e.sum())
    p = expected_probability
    p_hat = x / T

    log_restricted = (
        (T - x) * np.log(1 - p)
        + x * np.log(p)
    )

    if x == 0:
        log_unrestricted = 0.0
    elif x == T:
        log_unrestricted = 0.0
    else:
        log_unrestricted = (
            (T - x) * np.log(1 - p_hat)
            + x * np.log(p_hat)
        )

    lr = -2 * (log_restricted - log_unrestricted)
    p_value = chi2.sf(lr, 1)

    return {
        "Observations": T,
        "Exceptions": x,
        "Observed_Rate": p_hat,
        "LR_Statistic": lr,
        "p_value": p_value,
        "Result_at_5pct": "Reject H0" if p_value < 0.05 else "Do not reject H0"
    }

def christoffersen_test(exception_series):
    e = np.asarray(exception_series, dtype=int)

    previous = e[:-1]
    current = e[1:]

    n00 = int(np.sum((previous == 0) & (current == 0)))
    n01 = int(np.sum((previous == 0) & (current == 1)))
    n10 = int(np.sum((previous == 1) & (current == 0)))
    n11 = int(np.sum((previous == 1) & (current == 1)))

    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    pi0 = n01 / (n00 + n01)
    pi1 = n11 / (n10 + n11)

    def term(count, probability):
        if count == 0:
            return 0.0
        return count * np.log(probability)

    log_independent = (
        term(n00 + n10, 1 - pi)
        + term(n01 + n11, pi)
    )

    log_markov = (
        term(n00, 1 - pi0)
        + term(n01, pi0)
        + term(n10, 1 - pi1)
        + term(n11, pi1)
    )

    lr = -2 * (log_independent - log_markov)
    p_value = chi2.sf(lr, 1)

    return {
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "pi0": pi0,
        "pi1": pi1,
        "LR_Statistic": lr,
        "p_value": p_value,
        "Result_at_5pct": "Reject H0" if p_value < 0.05 else "Do not reject H0"
    }

kupiec_ewma = kupiec_test(ewma["Validation_Exception"])
christoffersen_ewma = christoffersen_test(ewma["Validation_Exception"])

ewma_test_summary = pd.DataFrame({
    "Test": [
        "Kupiec POF",
        "Christoffersen Independence"
    ],
    "Test_Statistic": [
        kupiec_ewma["LR_Statistic"],
        christoffersen_ewma["LR_Statistic"]
    ],
    "p_value": [
        kupiec_ewma["p_value"],
        christoffersen_ewma["p_value"]
    ],
    "Result_at_5pct": [
        kupiec_ewma["Result_at_5pct"],
        christoffersen_ewma["Result_at_5pct"]
    ]
})

combined = pd.read_csv(COMBINED_FILE)
combined["Date"] = pd.to_datetime(combined["Date"])
combined = combined.sort_values("Date").reset_index(drop=True)

required_columns = [
    "Date",
    "Portfolio_PnL",
    "Historical_VaR",
    "Parametric_VaR",
    "EWMA_VaR"
]

for column in required_columns:
    if column not in combined.columns:
        raise ValueError(f"Missing column: {column}")

model_columns = {
    "Historical": "Historical_VaR",
    "Parametric": "Parametric_VaR",
    "EWMA": "EWMA_VaR"
}

benchmark_rows = []
detailed_test_rows = []

for model, var_column in model_columns.items():
    exception_column = f"{model}_Exception"

    combined[exception_column] = (
        combined["Portfolio_PnL"] < -combined[var_column]
    )

    kupiec = kupiec_test(combined[exception_column])
    christoffersen = christoffersen_test(combined[exception_column])

    benchmark_rows.append({
        "Model": model,
        "Average_VaR": combined[var_column].mean(),
        "Maximum_VaR": combined[var_column].max(),
        "Actual_Exceptions": int(combined[exception_column].sum()),
        "Exception_Rate": combined[exception_column].mean(),
        "Kupiec_p_value": kupiec["p_value"],
        "Christoffersen_p_value": christoffersen["p_value"]
    })

    detailed_test_rows.append({
        "Model": model,
        "Kupiec_LR": kupiec["LR_Statistic"],
        "Kupiec_p_value": kupiec["p_value"],
        "Kupiec_Result": kupiec["Result_at_5pct"],
        "n00": christoffersen["n00"],
        "n01": christoffersen["n01"],
        "n10": christoffersen["n10"],
        "n11": christoffersen["n11"],
        "Christoffersen_LR": christoffersen["LR_Statistic"],
        "Christoffersen_p_value": christoffersen["p_value"],
        "Christoffersen_Result": christoffersen["Result_at_5pct"]
    })

benchmark = pd.DataFrame(benchmark_rows).set_index("Model")
detailed_tests = pd.DataFrame(detailed_test_rows)

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    combined["Date"],
    -combined["Historical_VaR"],
    label="Historical VaR",
    linewidth=1.1
)

ax.plot(
    combined["Date"],
    -combined["Parametric_VaR"],
    label="Parametric VaR",
    linewidth=1.1
)

ax.plot(
    combined["Date"],
    -combined["EWMA_VaR"],
    label="EWMA VaR",
    linewidth=1.1
)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("1-Day 99% VaR Model Comparison for $1M SPY Portfolio")
ax.set_xlabel("Date")
ax.set_ylabel("VaR Loss Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "var_model_comparison_full_period.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

covid = combined[
    (combined["Date"] >= "2020-02-01")
    & (combined["Date"] <= "2020-06-30")
].copy()

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    covid["Date"],
    -covid["Historical_VaR"],
    label="Historical VaR",
    linewidth=1.4
)

ax.plot(
    covid["Date"],
    -covid["Parametric_VaR"],
    label="Parametric VaR",
    linewidth=1.4
)

ax.plot(
    covid["Date"],
    -covid["EWMA_VaR"],
    label="EWMA VaR",
    linewidth=1.4
)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("VaR Model Comparison During COVID-19 Stress Period")
ax.set_xlabel("Date")
ax.set_ylabel("VaR Loss Threshold ($)")
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "var_model_comparison_covid.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

exception_summary.to_csv(
    VALIDATION_DIR / "ewma_exception_summary.csv",
    index=False
)

annual.to_csv(
    VALIDATION_DIR / "ewma_exceptions_by_year.csv"
)

ewma_test_summary.to_csv(
    VALIDATION_DIR / "ewma_statistical_tests.csv",
    index=False
)

benchmark.to_csv(
    VALIDATION_DIR / "model_benchmark_table.csv"
)

detailed_tests.to_csv(
    VALIDATION_DIR / "all_model_statistical_tests.csv",
    index=False
)

print("\nEWMA EXCEPTION SUMMARY")
print(exception_summary.to_string(index=False))

annual_display = annual.copy()
annual_display["Expected_Exceptions"] = annual_display["Expected_Exceptions"].round(2)
annual_display["Exception_Rate"] = annual_display["Exception_Rate"].map(lambda x: f"{x:.2%}")

print("\nANNUAL EWMA EXCEPTION BREAKDOWN")
print(annual_display.to_string())

test_display = ewma_test_summary.copy()
test_display["Test_Statistic"] = test_display["Test_Statistic"].map(lambda x: f"{x:.6f}")
test_display["p_value"] = test_display["p_value"].map(lambda x: f"{x:.6f}")

print("\nEWMA STATISTICAL TEST SUMMARY")
print(test_display.to_string(index=False))

benchmark_display = benchmark.copy()
benchmark_display["Average_VaR"] = benchmark_display["Average_VaR"].map(lambda x: f"${x:,.2f}")
benchmark_display["Maximum_VaR"] = benchmark_display["Maximum_VaR"].map(lambda x: f"${x:,.2f}")
benchmark_display["Exception_Rate"] = benchmark_display["Exception_Rate"].map(lambda x: f"{x:.2%}")
benchmark_display["Kupiec_p_value"] = benchmark_display["Kupiec_p_value"].map(lambda x: f"{x:.6f}")
benchmark_display["Christoffersen_p_value"] = benchmark_display["Christoffersen_p_value"].map(lambda x: f"{x:.6f}")

print("\nMODEL BENCHMARK TABLE")
print(benchmark_display.to_string())

print("\nDETAILED STATISTICAL RESULTS")
print(detailed_tests.to_string(index=False))

print("\nFILES SAVED TO")
print(VALIDATION_DIR)
print("\nValidation analysis completed successfully.")

<Figure size 1300x600 with 1 Axes>

<Figure size 1300x600 with 1 Axes>

<Figure size 1300x600 with 1 Axes>


EWMA EXCEPTION SUMMARY
                  Metric       Value
Backtesting Observations 3939.000000
     Expected Exceptions   39.390000
       Actual Exceptions   91.000000
 Expected Exception Rate    0.010000
 Observed Exception Rate    0.023102

ANNUAL EWMA EXCEPTION BREAKDOWN
      Observations  Expected_Exceptions  Actual_Exceptions Exception_Rate
Year                                                                     
2010             1                 0.01                  0          0.00%
2011           252                 2.52                  6          2.38%
2012           250                 2.50                  5          2.00%
2013           252                 2.52                  5          1.98%
2014           252                 2.52                 10          3.97%
2015           252                 2.52                  5          1.98%
2016           252                 2.52                  2          0.79%
2017           251                 2.51                